# Detecting Idioms in Sentence

To replace idioms in a sentence, we first have to ***detect*** and ***locate*** idioms.

There are two approaches to this:

1. Fuzzy matching

2. BERT (Bidirectional Encoder Representation from Transformers)

In [23]:
from rapidfuzz import process, fuzz
import spacy
import nltk
from lemminflect import getInflection
from idiom_parser import *
import re

nlp = spacy.load("en_core_web_sm")

idiomParser = Idioms()

def getBestDefinition(sentence: str):
    return idiomParser.find_best_definition(sentence)

def replaceIdiom(sentence, idiom, definition):
    if (definition.lower().startswith("to")):
        definition = definition[3:].strip()

    definition = definition.replace(".", "")

    doc = nlp(sentence)
    idiom_doc = nlp(idiom)
    def_doc = nlp(definition)

    print(def_doc[0].pos_)

    target_verb = None
    for token in doc:
        if (token.lemma_ == idiom_doc[0].lemma_):
            target_verb = token
            break
    
    if target_verb:
        tag = target_verb.tag_

        def_verb = def_doc[0].lemma_

        inflected_verb = getInflection(def_verb, tag=tag)[0]

        new_phrase = inflected_verb + " " + " ".join([t.text for t in def_doc[1:]])

        return sentence.replace(idiom, new_phrase).strip()
    
    return sentence.replace(idiom, definition).strip()

sentence = "He's chasing the american dream."
matches = idiomParser.find_idiom_matches(sentence)

for match in matches:
    result = replaceIdiom(sentence, match["text"], match["definition"])
    print(result)

[nltk_data] Downloading package brown to /Users/connor/nltk_data...
[nltk_data]   Package brown is already up-to-date!


DET
He's chasing the a philosophy that with hard work , courage and determination , anyone can prosper and achieve success.


In [17]:
import requests

def llm_fix_grammar(sentence, idiom_matches):
    for match in idiom_matches:
        idiom = match["text"]
        definition = match["definition"]

        prompt = f"""
        Original sentence: "{sentence}"
        Replace the phrase "{idiom}" with the definition "{definition}".
        Rewrite the sentence so that it is grammatically perfect and natural.
        Only return the corrected sentence. No explanation.
        """

        res = requests.post("http://localhost:11434/api/generate",
                                 json={
                                     "model": "llama3",
                                     "prompt": prompt,
                                     "stream": False
                                 })
        if (res.status_code == 200):
            sentence = res.json()["response"].strip()
        
    return sentence

sentence = "He's chasing the american dream."
matches = idiomParser.find_idiom_matches(sentence)

res = llm_fix_grammar(sentence, matches)
print(res)



He's chasing the philosophy that with hard work, courage, and determination, anyone can prosper and achieve success.


In [31]:
def reduce_single_present_tense(sentence: str):
    doc = nlp(sentence)

    new_tokens = []

    for token in doc:
        if (token.pos_ == "VERB" or token.pos_ == "AUX"):

            if (token.tag_ in ["VBD", "VBG", "VBN", "VBP"]):
                new_tokens.append(token.lemma_ + "s" if token.lemma_ != "be" else "is")
            else:
                new_tokens.append(token.text)

        elif (token.pos_ == "NOUN"):
            new_tokens.append(token.lemma_)

        else:
            new_tokens.append(token.text)
    res = " ".join(new_tokens)

    res = re.sub(r' \'', r"'", res)
    res = re.sub(r' \.', r'.', res)
    
    return res

sentence = "All these sunday drivers are causing so much traffic"
print(reduce_single_present_tense(sentence))

All these sunday driver is causes so much traffic


In [2]:
# from nltk.stem import WordNetLemmatizer
# from nltk.corpus import wordnet

# nltk.download("punkt")
# nltk.download("averaged_perceptron_tagger")
# nltk.download("wordnet")

# def singluar_present_tense(sentence: str):
#     lemmatizer = WordNetLemmatizer()

#     def get_wordnet_pos(word):
#         tag = nltk.pos_tag([word])[0][1][0].lower()

#         tag_dict = {"a": wordnet.ADJ, "n": wordnet.NOUN, "v": wordnet.VERB, "r": wordnet.ADJ}
#         return tag_dict.get(tag, wordnet.NOUN)
    
#     tokens = nltk.word_tokenize(sentence)
#     lemmatized_words = [lemmatizer.lemmatize(w, get_wordnet_pos(w)) for w in tokens]

#     return " ".join(lemmatized_words)

# print(singluar_present_tense(sentence))

In [1]:
from idiom_parser import *
import re
import nltk

IdiomParser = Idioms()

sentence = 'Replace the idioms in this sentence with their definition: "She was spilling the beans"'

# question, idiom_sent = sentence.split(":")

# idiom_sent = idiom_sent.replace('"', "")
# cleaned_sent = IdiomParser.reduce_single_present_tense(idiom_sent)
# cleaned_input = question + ': "' + cleaned_sent.strip() + '"'

# matches = IdiomParser.pick_out_idioms(cleaned_input)

# tokens = nltk.word_tokenize(sentence)
# for match in matches:
#     idiom, start, end = match

#     idiom_tokens = nltk.word_tokenize(idiom)

#     for i, j in enumerate(range(start, end)):
#         tokens[j] = idiom_tokens[i]
    
# sentence = " ".join(tokens)
# sentence = re.sub(r' ``', r'', sentence)
# sentence = re.sub(r' \'\'', r'', sentence)

print(IdiomParser.respond(sentence))


LLM Generated Sentence
Here's my best guess to replace your sentence: She was revealing a secret.


In [8]:
print(IdiomParser.idiom_df[IdiomParser.idiom_df["all_variations"].str.contains("spill the beans")])

KeyError: "None of [Index([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,\n       ...\n       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan],\n      dtype='float64', length=9721)] are in the [columns]"

In [1]:
import duckdb

PARQUET_PATH = "../Data/idiom_repository_replaced.parquet"

query = f"""
    CREATE TABLE
        idioms
    AS (
        SELECT *
        FROM '{PARQUET_PATH}'
    )
"""

duckdb.query(query)

In [6]:
query = f"""
    UPDATE
        idioms
    SET
        variations = ['under the hatch']
    WHERE
        idiom == 'under the hatches'
"""

duckdb.query(query)

In [3]:
from idiom_parser import *

df = duckdb.query("SELECT * FROM idioms").df()

IdiomParser = Idioms()

for _, row in df.iterrows():
    idiom = row["idiom"]
    variations = row["variations"]

    replacement = row["replacement"]

    singular_idiom = IdiomParser.reduce_single_present_tense(idiom)

    if (idiom.lower() != singular_idiom.lower()):
        query = f"""
            UPDATE
                idioms
            SET
                variations = ?
            WHERE
                idiom = ?
        """
        duckdb.query(query, params=[[singular_idiom], idiom])

In [ ]:
duckdb.query("""
    INSERT INTO idioms
        VALUES         
    """)

In [5]:
duckdb.query("COPY (SELECT * FROM idioms) TO 'idiom_repository_final.parquet' (FORMAT parquet)")